# User Modeling
## Goal 
- Building the first version of the review simulator

## importing dependencies

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from rouge_score import rouge_scorer as rouge_module


import warnings
warnings.filterwarnings('ignore')

In [2]:
#loading the data into the notebook 
df = pd.read_csv("/kaggle/input/datasets/ravirajbabasomane/amazon-reviews-2023/Amazon_reviews_2023.csv")

In [3]:
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True


## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [4]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (701528, 10)

Column dtypes:
rating                int64
title                object
text                 object
images               object
asin                 object
parent_asin          object
user_id              object
timestamp            object
helpful_vote          int64
verified_purchase      bool
dtype: object

Missing values:
rating                 0
title                160
text                 212
images                 0
asin                   0
parent_asin            0
user_id                0
timestamp              0
helpful_vote           0
verified_purchase      0
dtype: int64

Duplicate rows: 7275


### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [5]:
## understanding the schema of the review dataset 
df.iloc[0].to_dict()

{'rating': 5,
 'title': 'Such a lovely scent but not overpowering.',
 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
 'images': '[]',
 'asin': 'B00YQ6X8EO',
 'parent_asin': 'B00YQ6X8EO',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': '2020-05-05 14:08:48.923',
 'helpful_vote': 0,
 'verified_purchase': True}

In [6]:
#checking the columns in the data set 
df.columns.tolist()

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona

- the results here are quite worse to build the prototype i will use users with 5 or more reviews 

In [7]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

count    631986.000000
mean          1.110037
std           0.753202
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         165.000000
dtype: float64

In [8]:
# How many viable user?
five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")


The number of users with five reviews or more are 1620,
        The number of users with ten reviews or more are 330,
        while the number of users with twenty reviews or more are 117


In [9]:
# Distribution shape
user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

1     583553
2      39274
3       5713
4       1826
5        558
6        351
7        181
8        130
9         70
11        35
10        35
13        33
12        26
16        20
15        20
14        19
17        14
21         9
26         8
24         8
Name: count, dtype: int64

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [10]:
# Global rating distribution
df['rating'].value_counts().sort_index()

rating
1    102080
2     43034
3     56307
4     79381
5    420726
Name: count, dtype: int64

In [11]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

count    631986.000000
mean          3.948681
std           1.487903
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [12]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

count    48433.000000
mean         0.697705
std          0.918379
min          0.000000
25%          0.000000
50%          0.000000
75%          1.414214
max          2.828427
Name: rating, dtype: float64

In [13]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

count    1.620000e+03
mean    -1.484943e-02
std      2.539332e-01
min     -1.200000e+00
25%     -8.571429e-02
50%     -1.321078e-16
75%      5.714286e-02
max      1.100000e+00
Name: rating_slope, dtype: float64
trend
insufficient_data        630366
consistent                  734
increasingly_critical       475
increasingly_generous       411
Name: count, dtype: int64


In [14]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

"                      timestamp  rating  review_seq                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             text\n242833  2016-07-0

In [15]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f}   (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [16]:

"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

USER PROFILE: AHBWH2LBU3NFLD46GKJKIBAHKXEQ

 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.05   (std: 1.00)
   Avg review length: 104 words
   Rating breakdown: {1: 1, 2: 3, 3: 3, 4: 18, 5: 14}
   Rating trend    : ↑ more generous over time  (early avg: 3.89 → late avg: 4.20)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Unknown date  |  ⭐⭐⭐⭐ (4/5)  |  237 words
     Headline : Good alternative to travel emery boards
     I try to avoid using traditional files and emery boards
     on my nails. I take a lot of medications that make my
     nails weak and brittle, and any surface that's too
     abrasive wreaks havoc on my fingertips. I can't carry a
     full size glass nail file with me everywhere, so I was
     on the hunt for a travel sized file that could fit in
     my pocket, wallet, or purse. When these came up, I
     thought I'd give them a shot. The pros are the files
     are a great size, they'

In [17]:
# random user with 30+ reviews
user_history = profile_user(df)

Randomly selected user: AGFAOH3NMW2D7YV3QVZSTXMTSKIQ

USER PROFILE: AGFAOH3NMW2D7YV3QVZSTXMTSKIQ

 QUICK STATS
   Total reviews   : 47
   Avg rating      : 3.85   (std: 1.04)
   Avg review length: 99 words
   Rating breakdown: {1: 2, 2: 3, 3: 8, 4: 21, 5: 13}
   Rating trend    : ↓ more critical over time  (early avg: 4.17 → late avg: 3.54)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Unknown date  |  ⭐⭐⭐⭐ (4/5)  |  84 words
     Headline : Quick and Easy to Apply
     These were one of the easiest mask type products I have
     used.  The application process literally takes seconds.
     Peel off and place on face and they stay there.  I left
     them on for 15 minutes and could definitely tell the
     difference afterwards.  As the package describes there
     is a little residue left when you take them off which
     yu just dab in.  Over all I was pleased with the
     effect, but not pleased enough to want to p

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

In [18]:
# Review length distribution
df['text'].str.split().str.len().describe()

count    701316.000000
mean         32.760465
std          45.976779
min           0.000000
25%           8.000000
50%          19.000000
75%          40.000000
max        2585.000000
Name: text, dtype: float64

In [19]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

74629

In [20]:
# Avg text length per user
avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [21]:
display(avg_text_per_user)

user_id
AE222BBOVZIF42YOOPNBXL4UUMYA     52.0
AE222FP7YRNFCEQ2W3ZDIGMSYTLQ     44.0
AE222X475JC6ONXMIKZDFGQ7IAUA     14.0
AE222Y4WTST6BUZ4J5Y2H6QMBITQ    184.0
AE2232TEZOEWQLAFEX2NA6VBGMYQ     19.0
                                ...  
AHZZYVEU6QFMPFZ2HJUWR22SNK4A     11.0
AHZZZAK24AJ3JNBDUZJGHHWSRVAA    226.0
AHZZZJP24QUSB5XWW6MAXYBZZZSQ     33.0
AHZZZL7YQJA3RSA6PYK3WMFACYIQ    114.0
AHZZZSOTVOVACVK2WWXL4ITEAPIA     17.0
Name: text, Length: 631986, dtype: float64

In [22]:
#print(avg_text_per_user.value_counts())


In [23]:
# Verified purchase flag

df['verified_purchase'].value_counts()
# prefer verified = True

verified_purchase
True     634969
False     66559
Name: count, dtype: int64

## Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data 

In [24]:
import pandas as pd

def clean_amazon_reviews(df, 
                          min_reviews=20, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining               : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [25]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

AMAZON REVIEWS — CLEANING PIPELINE

▶ Starting shape: 701,528 rows × 11 columns

[1] Duplicate rows removed   : 0
    Remaining                : 701,528

[2] Duplicate columns removed: 0
    Remaining columns        : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'review_seq']

[3] Rows dropped (missing critical fields): 212
    Remaining                              : 701,316

[4] Rows dropped (empty / short reviews) : 456,456
    Min word count threshold             : 30 words
    Remaining                            : 244,860

[5] Rows dropped (unverified purchases)  : 41,244
    Remaining                            : 203,616

[6] Rows dropped (invalid ratings)       : 0
    Remaining                            : 203,616

[7] Users dropped (< 10 reviews)          : 192,293
    Viable users remaining               : 7
    Rows remaining                       : 100

[8] Sorted by user_id + timestamp ✓

✅ CLEAN

In [26]:
rich_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_seq,word_count
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,[],B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,1,158
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,[],B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,2,97
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,[],B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,3,137
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",[],B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,4,132
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,[],B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,5,124


In [27]:
rich_df.to_csv('rich_users.csv', index=False)

# Part 2 Build a Persona 

## Persona style 

In [28]:

import re

# ── STEP 1: STATISTICAL PERSONA (no LLM needed) ─────────────────

def extract_statistical_persona(df):
    """
    Extracts measurable traits per user from their review history.
    No LLM call yet — pure signal extraction from data.
    """

    def rating_trend(ratings):
        """slope of ratings over time — positive = more generous, negative = more critical"""
        if len(ratings) < 4:
            return 0.0
        x = np.arange(len(ratings))
        slope = np.polyfit(x, ratings, 1)[0]
        return round(float(slope), 4)

    def writing_style(texts):
        """extract fingerprint signals from review text"""
        texts = [str(t) for t in texts if len(str(t)) > 10]
        if not texts:
            return {}
        
        all_sentences = []
        all_words     = []
        openers       = []
        
        for text in texts:
            sentences = re.split(r'[.!?]+', text)
            sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
            words     = text.lower().split()
            
            all_sentences.extend(sentences)
            all_words.extend(words)
            if words:
                openers.append(words[0])  # first word of each review
        
        avg_sentence_len = np.mean([len(s.split()) for s in all_sentences]) if all_sentences else 0
        vocab_richness   = len(set(all_words)) / len(all_words) if all_words else 0  # type-token ratio
        avg_review_len   = np.mean([len(t.split()) for t in texts])
        
        # is this person verbose or terse?
        verbosity = "verbose" if avg_review_len > 80 \
                    else "moderate" if avg_review_len > 40 \
                    else "terse"
        
        # most common opener words (excluding stopwords)
        stopwords = {'i', 'the', 'this', 'a', 'an', 'it', 'my', 'we', 'so'}
        meaningful_openers = [w for w in openers if w not in stopwords]
        top_opener = meaningful_openers[0] if meaningful_openers else "unknown"
        
        return {
            "avg_sentence_len"  : round(avg_sentence_len, 1),
            "vocab_richness"    : round(vocab_richness, 3),
            "avg_review_len"    : round(avg_review_len, 1),
            "verbosity"         : verbosity,
            "common_opener"     : top_opener
        }

    def complaint_signals(texts):
        """detect recurring complaint or praise patterns"""
        all_text = " ".join([str(t).lower() for t in texts])
        
        signals = {
            "mentions_price"    : bool(re.search(r'\b(price|expensive|cheap|cost|worth|value|money)\b', all_text)),
            "mentions_quality"  : bool(re.search(r'\b(quality|durable|broke|lasted|cheap|flimsy|solid)\b', all_text)),
            "mentions_shipping" : bool(re.search(r'\b(shipping|delivery|arrived|package|days|late|fast)\b', all_text)),
            "mentions_service"  : bool(re.search(r'\b(service|support|help|staff|seller|response)\b', all_text)),
            "uses_caps"         : bool(re.search(r'\b[A-Z]{3,}\b', " ".join([str(t) for t in texts]))),
            "uses_exclamation"  : " ".join([str(t) for t in texts]).count("!") > len(texts)
        }
        return signals

    def rating_profile(ratings):
        """classify this user's rating personality"""
        avg = np.mean(ratings)
        std = np.std(ratings)
        
        if avg >= 4.2:
            generosity = "generous"
        elif avg <= 2.8:
            generosity = "harsh"
        else:
            generosity = "balanced"
        
        consistency = "consistent" if std < 0.8 else "variable"
        
        return {
            "generosity"   : generosity,
            "consistency"  : consistency,
            "never_gives_5": int(5 not in ratings),
            "never_gives_1": int(1 not in ratings)
        }

    # ── AGGREGATE PER USER ───────────────────────────────────────
    rows = []

    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('timestamp') if 'timestamp' in group.columns else group

        ratings  = group['rating'].tolist()
        texts    = group['text'].tolist()

        style    = writing_style(texts)
        signals  = complaint_signals(texts)
        profile  = rating_profile(ratings)

        row = {
            "user_id"      : user_id,

            # --- rating stats ---
            "review_count"    : len(ratings),
            "avg_rating"      : round(np.mean(ratings), 2),
            "rating_std"      : round(np.std(ratings), 2),
            "rating_trend"    : rating_trend(ratings),

            # --- rating profile ---
            "generosity"      : profile["generosity"],
            "consistency"     : profile["consistency"],
            "never_gives_5"   : profile["never_gives_5"],
            "never_gives_1"   : profile["never_gives_1"],

            # --- writing style ---
            "verbosity"       : style.get("verbosity"),
            "avg_review_len"  : style.get("avg_review_len"),
            "vocab_richness"  : style.get("vocab_richness"),
            "avg_sentence_len": style.get("avg_sentence_len"),
            "common_opener"   : style.get("common_opener"),

            # --- complaint / praise signals ---
            "mentions_price"  : signals["mentions_price"],
            "mentions_quality": signals["mentions_quality"],
            "mentions_shipping": signals["mentions_shipping"],
            "mentions_service": signals["mentions_service"],
            "uses_caps"       : signals["uses_caps"],
            "uses_exclamation": signals["uses_exclamation"],

            # --- sample reviews for LLM trait extraction (next step) ---
            "sample_reviews"  : texts[:5],
            "reviewed_items"  : group['asin'].tolist() if 'asin' in group.columns else []
        }
        rows.append(row)

    persona_df = pd.DataFrame(rows)
    print(f" Statistical personas extracted for {len(persona_df):,} users")
    print(f"   Columns: {list(persona_df.columns)}")
    return persona_df


# ── RUN IT ───────────────────────────────────────────────────────
persona_df_claude = extract_statistical_persona(rich_df)

 Statistical personas extracted for 7 users
   Columns: ['user_id', 'review_count', 'avg_rating', 'rating_std', 'rating_trend', 'generosity', 'consistency', 'never_gives_5', 'never_gives_1', 'verbosity', 'avg_review_len', 'vocab_richness', 'avg_sentence_len', 'common_opener', 'mentions_price', 'mentions_quality', 'mentions_shipping', 'mentions_service', 'uses_caps', 'uses_exclamation', 'sample_reviews', 'reviewed_items']


In [29]:
persona_df_claude.head()

,user_id,review_count,avg_rating,rating_std,rating_trend,generosity,consistency,never_gives_5,never_gives_1,verbosity,...,avg_sentence_len,common_opener,mentions_price,mentions_quality,mentions_shipping,mentions_service,uses_caps,uses_exclamation,sample_reviews,reviewed_items
0,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,11,4.64,0.48,0.0455,generous,consistent,0,1,verbose,...,16.1,these,True,True,True,True,True,False,[This makeup is crazy. It comes out of the bo...,"[B07VNQ4G13, B07VML1QZC, B07X1PH59J, B07XNYVBY..."
1,AEYKTZXAWOPJG5MGGMKBLRJR6Q3A_2,21,4.95,0.21,0.0052,generous,consistent,0,1,moderate,...,41.9,impressive,True,True,True,False,False,False,[Impressive device. Operates on by charge on u...,"[B07WMFRDZ3, B07QYD1J5C, B07XRJC4TY, B07YSC7V2..."
2,AFR4BTNWATNG7O4SOME4XK6NNBYQ,10,4.80,0.40,0.0848,generous,consistent,0,1,moderate,...,11.8,what,False,False,True,False,True,False,[The bar is rather small..but the jasmine is o...,"[B0013EHI6G, B00CBY2W4U, B004FO8KPE, B00TJE4JU..."
3,AFSCJNRG4BAGAB37REW4XCDPE6XA,11,5.00,0.00,0.0000,generous,consistent,0,1,moderate,...,10.7,tired,True,False,False,False,False,False,[My first experience with a shaver Like this a...,"[B07RG332M3, B07QQQXZ4S, B07Q3BSDC8, B07V6BK6H..."
4,AFUBMCVI5J6G4F2RFQGTDRXLE6VQ,10,3.40,1.11,-0.0364,balanced,variable,0,1,verbose,...,11.2,fantastic,True,False,True,True,True,False,[fantastic value for fake hair. perfect color...,"[B00D93WF26, B00IJVVBG4, B089XX8C33, B08D3HBC3..."


## Prompt Building

In [30]:
def build_user_prompt(user_row, item_asin, item_title, item_description, nigerian_mode=False ):
    """
    Builds a persona-conditioned prompt for review simulation.
    Designed for Gemini API. Uses chain-of-thought before generation.
    
    Parameters:
        user_row         : one row from persona_df
        item_asin        : product ID
        item_title       : product name
        item_description : product description
        nigerian_mode    : inject Nigerian cultural conditioning layer
    """

    # ── 1. BUILD TRAIT SUMMARY ───────────────────────────────────
    # translate extracted traits into natural language descriptions
    # so the LLM gets character, not just numbers

    trait_lines = []

    # rating personality
    generosity   = user_row.get('generosity', 'balanced')
    consistency  = user_row.get('consistency', 'consistent')
    avg_rating   = user_row.get('avg_rating', 3.0)
    rating_std   = user_row.get('rating_std', 1.0)
    rating_trend = user_row.get('rating_trend', 0.0)

    trait_lines.append(f"- They are a {generosity} rater — their average rating is {avg_rating:.1f}/5")
    trait_lines.append(f"- They are {consistency} in their ratings (std dev: {rating_std:.2f})")

    if user_row.get('never_gives_5'):
        trait_lines.append("- They NEVER give 5 stars — even for things they like")
    if user_row.get('never_gives_1'):
        trait_lines.append("- They have never given 1 star — even when disappointed")
    if rating_trend > 0.05:
        trait_lines.append("- Their ratings have been trending upward — they are becoming more generous over time")
    elif rating_trend < -0.05:
        trait_lines.append("- Their ratings have been trending downward — they are becoming more critical over time")

    # writing style
    verbosity        = user_row.get('verbosity', 'moderate')
    avg_review_len   = int(user_row.get('avg_review_len', 50))
    vocab_richness   = user_row.get('vocab_richness', 0.5)
    avg_sentence_len = user_row.get('avg_sentence_len', 15)
    common_opener    = user_row.get('common_opener', '')

    trait_lines.append(f"- Writing style: {verbosity} — they write around {avg_review_len} words per review")
    trait_lines.append(f"- Sentence length: avg {avg_sentence_len:.0f} words per sentence")
    if vocab_richness > 0.7:
        trait_lines.append("- They use rich, varied vocabulary — rarely repeat the same words")
    elif vocab_richness < 0.4:
        trait_lines.append("- They use simple, repetitive vocabulary — plain everyday language")
    if common_opener:
        trait_lines.append(f"- They often start their reviews with words like: '{common_opener}'")

    # complaint / praise signals
    if user_row.get('mentions_price'):
        trait_lines.append("- Price and value for money is a recurring theme in their reviews")
    if user_row.get('mentions_quality'):
        trait_lines.append("- Product quality and durability is something they always comment on")
    if user_row.get('mentions_shipping'):
        trait_lines.append("- They frequently mention shipping speed and delivery experience")
    if user_row.get('mentions_service'):
        trait_lines.append("- Customer service experience often appears in their reviews")
    if user_row.get('uses_exclamation'):
        trait_lines.append("- They use exclamation marks frequently — expressive and emotional tone")
    if user_row.get('uses_caps'):
        trait_lines.append("- They occasionally use ALL CAPS for emphasis")

    traits_block = "\n".join(trait_lines)

    # ── 2. FORMAT SAMPLE REVIEWS ─────────────────────────────────
    samples = user_row.get('sample_reviews', [])
    if samples:
        sample_block = "\n\n".join([
            f"  Past review {i+1}:\n  \"{str(r).strip()}\""
            for i, r in enumerate(samples[:4])  # max 4 samples
        ])
    else:
        sample_block = "  No sample reviews available."

    # ── 3. NIGERIAN CONDITIONING LAYER ───────────────────────────
    nigerian_block = ""
    if nigerian_mode:
        nigerian_block = """
IMPORTANT CULTURAL CONTEXT:
This reviewer is Nigerian. Their review should naturally reflect Nigerian consumer behaviour:
- They are price-conscious and often compare value to local alternatives
- They may reference familiar Nigerian brands, experiences, or comparisons ("like the ones in Shoprite", "better than what you find for Ikeja")
- They may code-switch lightly — mixing standard English with Pidgin phrases naturally (e.g. "e good sha", "I no go lie", "the thing dey work")
- Do NOT force Pidgin on every sentence — use it sparingly and naturally, the way an educated Nigerian would
- They may mention community or family context ("my whole family", "my colleagues at work")
- Scepticism about product claims is common — they may note what surprised them vs expectation
"""

    # ── 4. RATING ANCHOR ─────────────────────────────────────────
    # give the LLM a statistical prior so rating prediction stays
    # close to this user's actual behaviour — reduces RMSE significantly
    low  = max(1, round(avg_rating - rating_std))
    high = min(5, round(avg_rating + rating_std))
    rating_anchor = f"Based on their history, their rating will most likely fall between {low} and {high} stars."

    # ── 5. ASSEMBLE THE FULL PROMPT ──────────────────────────────
    prompt = f"""You are simulating the review behaviour of a specific, real Amazon user.
Your job is NOT to write a good review — your job is to write the review THIS SPECIFIC PERSON would write.
Stay in character throughout. Do not default to generic reviewer behaviour.

═══════════════════════════════════════════
WHO THIS PERSON IS
═══════════════════════════════════════════
{traits_block}

═══════════════════════════════════════════
SAMPLES OF HOW THEY ACTUALLY WRITE
═══════════════════════════════════════════
{sample_block}
{nigerian_block}
═══════════════════════════════════════════
THE PRODUCT THEY ARE REVIEWING (UNSEEN)
═══════════════════════════════════════════
Product title      : {item_title}
Product ID (ASIN)  : {item_asin}
Product description: {item_description}

═══════════════════════════════════════════
INSTRUCTIONS
═══════════════════════════════════════════
Before writing the review, reason briefly:
- What would this person notice about this product given their traits?
- What would they likely complain about or praise?
- What rating does their history suggest they would give?

{rating_anchor}

Then produce your output in EXACTLY this format — nothing before, nothing after:

REASONING: [2-3 sentences of internal reasoning about this user and this product]
RATING: [single integer 1-5]
TITLE: [short review headline in their voice]
REVIEW: [full review text in their voice and style]
""".strip()

    return prompt

In [31]:
# use the kaggle secret enviroment to secure my api key 
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

In [32]:
import google.generativeai as genai

genai.configure(api_key= GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

def simulate_review(user_row, item_asin, item_title, item_description, nigerian_mode=False):
    """
    Full pipeline: build prompt → call Gemini → parse output
    """
    prompt = build_user_prompt(
        user_row, item_asin, item_title, 
        item_description, nigerian_mode
    )
    
    response = model.generate_content(prompt)
    raw      = response.text.strip()
    
    # ── PARSE THE STRUCTURED OUTPUT ──────────────────────────
    result = {
        "user_id"   : user_row['user_id'],
        "asin"      : item_asin,
        "raw_output": raw
    }
    
    for field in ['REASONING', 'RATING', 'TITLE', 'REVIEW']:
        pattern = rf"{field}:\s*(.*?)(?=\n[A-Z]+:|$)"
        match   = re.search(pattern, raw, re.DOTALL)
        result[field.lower()] = match.group(1).strip() if match else None
    
    # cast rating to int safely
    try:
        result['rating'] = int(result['rating'])
    except (TypeError, ValueError):
        result['rating'] = round(user_row['avg_rating'])  # fallback to user avg
    
    return result

In [33]:
# grab one user from persona_df
test_user = persona_df_claude.iloc[0]

result = simulate_review(
    user_row         = test_user,
    item_asin        = "B09XYZ123",
    item_title       = "Wireless Bluetooth Earbuds",
    item_description = "Noise cancelling earbuds with 24hr battery life",
    nigerian_mode    = True
)

print("REASONING :", result['reasoning'])
print("RATING    :", result['rating'])
print("TITLE     :", result['title'])
print("REVIEW    :", result['review'])

REASONING : This user typically rates very high, so a 5-star review is likely if the product performs well and offers good value. They will focus on the key claims (noise cancelling, battery life) and compare the value to local alternatives. Shipping speed and product durability are also key points for them.
RATING    : 5
TITLE     : These Earbuds Are Really Good For The Price!
REVIEW    : These wireless Bluetooth earbuds are really good, I no go lie. When I saw they said noise cancelling, I was a bit skeptical, because most of them don't work well, but these ones actually block out a lot of noise. It works VERY WELL for my commute. The sound quality is also crisp and clear, much better than some of the ones my colleagues bought from Ikeja.

Battery life is another big one for me. They said 24 hours and I have used them for a full day without needing to charge, so the claim is accurate. This is super important. The shipping was also very fast, it came quicker than I thought it would he

In [34]:
!pip install rouge-score -q

In [36]:
# Cell 2 — Build Real Item Lookup (fixed)
item_lookup = df.groupby('asin').agg(
    item_title      = ('title', 'first'),
    avg_item_rating = ('rating', 'mean'),
    description     = ('text', lambda x: ' '.join(
        [str(i) for i in list(x)[:2] if pd.notna(i)]  # ← fix: skip NaN, force str
    ))
).reset_index()

print(f"Item lookup built: {len(item_lookup):,} items")
item_lookup.head(3)

Item lookup built: 115,709 items


,asin,item_title,avg_item_rating,description
0,0005946468,Five Stars,5.000000,great
1,0123034892,Five Stars,5.000000,Good product
2,0124784577,Que cumplió con mis expectativas de higiene pa...,4.333333,La manera de entrega poca seguridad de que lle...


In [39]:
def get_unseen_item(user_id, df, item_lookup):
    """Get a real item from dataset that this user hasn't reviewed"""
    reviewed_asins = set(df[df['user_id'] == user_id]['asin'])
    unseen = item_lookup[~item_lookup['asin'].isin(reviewed_asins)]
    return unseen.sample(1).iloc[0]

# Test it
test_user = persona_df_claude.iloc[0]
unseen_item = get_unseen_item(test_user['user_id'], df, item_lookup)
print(f"Unseen item for user: {unseen_item['item_title']}")
print(f"ASIN: {unseen_item['asin']}")

Unseen item for user: Great product!!
ASIN: B086KSKKHV


In [43]:
# Now simulate with a REAL beauty product, not made-up earbuds
result = simulate_review(
    user_row         = test_user,
    item_asin        = unseen_item['asin'],
    item_title       = unseen_item['item_title'],
    item_description = unseen_item['description'][:200],
    nigerian_mode    = True
)

print("REASONING :", result['reasoning'])
print("RATING    :", result['rating'])
print("TITLE     :", result['title'])
print("REVIEW    :", result['review'])

REASONING : The user is a generous rater, so despite significant product flaws (absorbency, residue), the positives (reusability, sale price, personal liking) will push the rating high. Their verbose, simple style, focus on value, durability, and Nigerian context (price comparison, light Pidgin) will be prominent.
RATING    : 4
TITLE     : Good Reusable Pads, But Absorbency Could Be Better
REVIEW    : These reusable pads are good, and I am happy I bought these. I got them on sale, and the price was good. I no go lie, I LOVE that they are reusable. This means I do not have to keep buying disposables from places like Shoprite. It helps me save money, and that is VERY important for my family. The quality seems good too; these pads look like they will last a long time, which is great for durability.

However, I must say the absorbency was not as strong as I hoped. This pad was not very absorbent, and my face still felt like it had some residue after washing. It just did not take off everyt

In [44]:

scorer = rouge_module.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def evaluate_pipeline(persona_df_claude, rich_df, n_users=30, nigerian_mode=False):
    results = []

    for _, user_row in persona_df_claude.head(n_users).iterrows():
        user_reviews = rich_df[
            rich_df['user_id'] == user_row['user_id']
        ].sort_values('timestamp')

        if len(user_reviews) < 4:
            continue

        # Hold out last review, train on the rest
        holdout  = user_reviews.iloc[-1]
        history  = user_reviews.iloc[:-1]

        temp_row = user_row.copy()
        temp_row['sample_reviews'] = list(history['text'])[:3]

        try:
            result = simulate_review(
                temp_row,
                holdout['asin'],
                holdout['title'],
                str(holdout['text'])[:200],
                nigerian_mode=nigerian_mode
            )
        except Exception as e:
            print(f"Skipping user {user_row['user_id'][:10]} — error: {e}")
            continue

        true_rating = holdout['rating']
        pred_rating = result['rating'] or round(user_row['avg_rating'])

        rouge_scores = scorer.score(
            str(holdout['text']),
            str(result['review'] or "")
        )

        results.append({
            'user_id'     : user_row['user_id'],
            'true_rating' : true_rating,
            'pred_rating' : pred_rating,
            'rating_error': abs(true_rating - pred_rating),
            'rouge1'      : rouge_scores['rouge1'].fmeasure,
            'rougeL'      : rouge_scores['rougeL'].fmeasure,
            'generated'   : result['review'],
            'true_review' : holdout['text']
        })

    results_df = pd.DataFrame(results)
    rmse = np.sqrt((results_df['rating_error'] ** 2).mean())

    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Users evaluated : {len(results_df)}")
    print(f"Rating RMSE     : {rmse:.4f}")
    print(f"ROUGE-1         : {results_df['rouge1'].mean():.4f}")
    print(f"ROUGE-L         : {results_df['rougeL'].mean():.4f}")
    print("=" * 50)

    return results_df

results = evaluate_pipeline(persona_df_claude, rich_df, n_users=30)

EVALUATION RESULTS
Users evaluated : 7
Rating RMSE     : 0.7559
ROUGE-1         : 0.4469
ROUGE-L         : 0.2656
